# License Plate Detection and Recognition

## Phase 0: Setup & Environment

In [ ]:
# Cell 1: Install packages and setup environment
!pip install -q ultralytics matplotlib opencv-python pyyaml gdown pandas

import os
import glob
import time
import shutil
import random
import yaml
from PIL import Image
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from ultralytics import YOLO

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active device: {device}")

#### Download and Extract Datasets

In [ ]:
# Cell 2: Download and extract datasets directly into Kaggle
# Clean up any broken files from previous runs
!rm -rf /kaggle/working/LPD.zip /kaggle/working/LPR.zip /kaggle/working/LPD_FILES /kaggle/working/LPR_FILES

print("Downloading LPD dataset...")
!gdown --fuzzy "https://drive.google.com/uc?id=1UBV930A9pRhVwop-WunmP38th2XwEumx" -O /kaggle/working/LPD.zip

print("Downloading LPR dataset...")
!gdown --fuzzy "https://drive.google.com/uc?id=1Me3Bk1mmITaf5QQE5Kvk1E1d8xaqsnhf" -O /kaggle/working/LPR.zip

print("Extracting datasets...")
!mkdir -p /kaggle/working/LPD_FILES
!unzip -q -o /kaggle/working/LPD.zip -d /kaggle/working/LPD_FILES

!mkdir -p /kaggle/working/LPR_FILES
!unzip -q -o /kaggle/working/LPR.zip -d /kaggle/working/LPR_FILES
print("Done! Ready for splitting.")

## Phase 1: License Plate Detection (YOLO)

#### 0. Directory Split

In [ ]:
base_dir = '/kaggle/working/LPD_FILES'
images_dir = os.path.join(base_dir, 'images')
labels_dir = os.path.join(base_dir, 'labels')

train_images = os.path.join(images_dir, 'train')
val_images = os.path.join(images_dir, 'val')
train_labels = os.path.join(labels_dir, 'train')
val_labels = os.path.join(labels_dir, 'val')

for d in [train_images, val_images, train_labels, val_labels]:
    os.makedirs(d, exist_ok=True)

all_images = [f for f in os.listdir(images_dir) if os.path.isfile(os.path.join(images_dir, f))]

random.seed(42)
random.shuffle(all_images)
split_idx = int(0.8 * len(all_images))

train_files = all_images[:split_idx]
val_files = all_images[split_idx:]

def move_dataset_files(file_list, split_type):
    img_dest = train_images if split_type == 'train' else val_images
    lbl_dest = train_labels if split_type == 'train' else val_labels

    for file in file_list:
        src_img = os.path.join(images_dir, file)
        shutil.move(src_img, os.path.join(img_dest, file))
        
        filename_no_ext = os.path.splitext(file)[0]
        label_file = filename_no_ext + '.txt'
        src_lbl = os.path.join(labels_dir, label_file)
        
        if os.path.exists(src_lbl):
            shutil.move(src_lbl, os.path.join(lbl_dest, label_file))

move_dataset_files(train_files, 'train')
move_dataset_files(val_files, 'val')

print(f"Dataset successfully split: {len(train_files)} files in 'train', {len(val_files)} files in 'val'.")

#### 1. Configure Dataset and `data.yaml`

In [ ]:
yaml_content = {
    'path': '/kaggle/working/LPD_FILES',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': {0: 'license_plate'}
}

yaml_path = "/kaggle/working/data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("Updated data.yaml:")
!cat {yaml_path}

#### 2. Train and Validate YOLO

In [ ]:
yolo_model = YOLO("yolov8n.pt")

# Kaggle will save automatically to /kaggle/working/runs/detect/lpd_yolo
yolo_results = yolo_model.train(
    data=yaml_path,
    epochs=25,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    name="lpd_yolo",
    save=True,
    plots=True
)

best_lpd_path = "/kaggle/working/runs/detect/lpd_yolo/weights/best.pt"
best_lpd_model = YOLO(best_lpd_path)
print(f"Best YOLO weights saved at: {best_lpd_path}")